## Install and import required libraries

In [1]:
# !pip install scikit-learn
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install xgboost
# !pip install pyarrow

In [2]:
import pandas as pd
import numpy as np
import glob
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

In [3]:
#load single file to look at contents
df = pd.read_csv('data/ids_0.csv')

#get names of columns
print(df.columns)

#display data types
print(df.dtypes)

#disply dataframe head
df.head()

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,53,61205,4,2,136,428,34,34,34.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,222,2,2,90,172,45,45,45.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,53,23759,2,2,70,126,35,35,35.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,401,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,57406,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
#combine all csv, json, and parquet files into single dataframe

def extract() -> pd.DataFrame:
    #main dataframe that everything will be concatenated to
    data = pd.DataFrame()

    #extract CSV files
    for csvfile in glob.glob('data/*.csv'):
        tmp_df = pd.read_csv(csvfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #extract JSON files
    for jsonfile in glob.glob('data/*.json'):
        tmp_df = pd.read_json(jsonfile, lines=True)
        data = pd.concat([data, tmp_df], ignore_index=True)

    #extract Parquet files
    for parquetfile in glob.glob('data/*.parquet'):
        tmp_df = pd.read_parquet(parquetfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #return combined dataframe
    return data

In [5]:
#call extract function to combine all files into single dataframe
df = extract()

print("Shape of combined dataframe:", df.shape)

df.head()

Shape of combined dataframe: (63129, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,55109,17,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,113594958,4,4,152,362,45,31,38.0,8.082904,...,32,240.0,0.0,240,240,114000000.0,0.0,114000000,114000000,BENIGN
2,53,30485,1,1,81,209,81,81,81.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,53,30445,1,1,53,81,53,53,53.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,53,70860,1,1,56,72,56,56,56.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [6]:
#verify the extracted data
#remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()  

#check label distribution
print(df['Label'].value_counts())

Label
DoS Hulk            31027
DoS GoldenEye       20586
BENIGN               6006
DoS Slowhttptest     5499
Heartbleed             11
Name: count, dtype: int64


In [8]:
#Tranform 

def transform(df: pd.DataFrame) -> pd.DataFrame:

    #remove Heartbleed rows since its not a DOS attack
    df = df[df['Label'] != 'Heartbleed']

    #remove any duplicate rows
    df = df.drop_duplicates()

    #remap labels (benign stays but all dos attacks get remapped to "attack")
    df['Label'] = df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'attack')

    #replace infinite values with NaN and then drop rows with NaN values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    return df


In [9]:
#call transform function to clean the data
df = transform(df)
print("Shape of transformed dataframe:", df.shape)

#check label distribution after transformation
print(df['Label'].value_counts())

Shape of transformed dataframe: (49348, 79)
Label
attack    44402
BENIGN     4946
Name: count, dtype: int64


In [ ]:
#load the data into a csv file
df.to_csv('data/cleaned_data.csv', index=False)

## Data Analysis and Preprocessing

Now that we have cleaned data, we need to:
1. Perform exploratory data analysis (EDA)
2. Preprocess features (scaling, splitting)
3. Engineer features (remove low-variance, highly correlated features)
4. Train machine learning models
5. Evaluate and compare model performance

In [ ]:
# Load the cleaned data
df = pd.read_csv('data/cleaned_data.csv')

print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

# Dataset shape
print(f"\nDataset shape: {df.shape}")
print(f"Number of features: {df.shape[1] - 1}")  # -1 for label column
print(f"Number of samples: {df.shape[0]}")

# Label distribution
print("\n" + "="*60)
print("LABEL DISTRIBUTION:")
print("="*60)
print(df['Label'].value_counts())
print(f"\nClass balance: {df['Label'].value_counts(normalize=True)}")

# Check for missing values
print("\n" + "="*60)
print("MISSING VALUES:")
print("="*60)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values found ✓")
else:
    print(missing[missing > 0])

# Display first few rows
print("\n" + "="*60)
print("FIRST 20 ROWS (for submission requirement):")
print("="*60)
df.head(20)

In [ ]:
# Visualize label distribution
plt.figure(figsize=(8, 5))
df['Label'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Distribution of Benign vs Attack Traffic', fontsize=14, fontweight='bold')
plt.xlabel('Traffic Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBenign traffic: {df[df['Label'] == 'BENIGN'].shape[0]:,} samples")
print(f"Attack traffic: {df[df['Label'] == 'attack'].shape[0]:,} samples")
print(f"Ratio (attack/benign): {df[df['Label'] == 'attack'].shape[0] / df[df['Label'] == 'BENIGN'].shape[0]:.2f}")

### Key Observations from EDA

In [ ]:
# Separate features and labels
X = df.drop('Label', axis=1)
y = df['Label']

# Convert labels to binary (0 = BENIGN, 1 = attack)
y_binary = (y == 'attack').astype(int)

print("="*60)
print("FEATURE AND LABEL PREPARATION")
print("="*60)
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y_binary.shape}")
print(f"\nLabel encoding:")
print(f"  0 = BENIGN")
print(f"  1 = attack")
print(f"\nClass distribution:")
print(y_binary.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 75% train, 25% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary,
    test_size=0.25,
    random_state=42,
    stratify=y_binary  # Maintain class balance in both sets
)

print("="*60)
print("TRAIN/TEST SPLIT")
print("="*60)
print(f"Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set label distribution:")
print(y_train.value_counts())
print(f"\nTesting set label distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("="*60)
print("FEATURE SCALING (StandardScaler)")
print("="*60)
print("Scaled training and testing features to have mean=0, std=1")
print(f"Training set scaled shape: {X_train_scaled.shape}")
print(f"Testing set scaled shape: {X_test_scaled.shape}")
print("\nSample statistics after scaling (training set):")
print(X_train_scaled.describe().iloc[:2, :5])  # Show first 5 features

In [ ]:
# Remove low-variance features
variance_threshold = 0.01
feature_variances = X_train_scaled.var()
low_variance_features = feature_variances[feature_variances < variance_threshold].index.tolist()

print("="*60)
print("FEATURE ENGINEERING: VARIANCE FILTERING")
print("="*60)
print(f"Features with variance < {variance_threshold}:")
if len(low_variance_features) > 0:
    print(f"Found {len(low_variance_features)} low-variance features:")
    for feat in low_variance_features:
        print(f"  - {feat}: variance = {feature_variances[feat]:.6f}")

    # Remove from both sets
    X_train_scaled = X_train_scaled.drop(columns=low_variance_features)
    X_test_scaled = X_test_scaled.drop(columns=low_variance_features)
    print(f"\nRemoved {len(low_variance_features)} features")
else:
    print("No low-variance features found ✓")

print(f"\nFeatures remaining: {X_train_scaled.shape[1]}")

In [ ]:
# Remove highly correlated features
correlation_threshold = 0.95
corr_matrix = X_train_scaled.corr().abs()

# Get upper triangle of correlation matrix
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find features with correlation > threshold
high_corr_features = [column for column in upper_tri.columns
                      if any(upper_tri[column] > correlation_threshold)]

print("="*60)
print("FEATURE ENGINEERING: CORRELATION FILTERING")
print("="*60)
print(f"Removing features with correlation > {correlation_threshold}")

if len(high_corr_features) > 0:
    print(f"Found {len(high_corr_features)} highly correlated features:")
    for feat in high_corr_features[:10]:  # Show first 10
        print(f"  - {feat}")
    if len(high_corr_features) > 10:
        print(f"  ... and {len(high_corr_features) - 10} more")

    # Remove from both sets
    X_train_scaled = X_train_scaled.drop(columns=high_corr_features)
    X_test_scaled = X_test_scaled.drop(columns=high_corr_features)
    print(f"\nRemoved {len(high_corr_features)} features")
else:
    print("No highly correlated features found")

print(f"\nFinal feature count: {X_train_scaled.shape[1]}")

In [ ]:
# Save processed datasets
X_train_scaled.to_csv('data/X_train_processed.csv', index=False)
X_test_scaled.to_csv('data/X_test_processed.csv', index=False)
y_train.to_csv('data/y_train_processed.csv', index=False, header=['Label'])
y_test.to_csv('data/y_test_processed.csv', index=False, header=['Label'])

print("="*60)
print("PROCESSED DATA SAVED")
print("="*60)
print("Saved files:")
print("  - data/X_train_processed.csv")
print("  - data/X_test_processed.csv")
print("  - data/y_train_processed.csv")
print("  - data/y_test_processed.csv")

## Model Selection and Training

We will train and evaluate 5 different machine learning models:
1. **Logistic Regression** - Simple baseline
2. **Decision Tree** - Non-linear classifier
3. **Random Forest** - Ensemble method
4. **Support Vector Machine (SVM)** - Effective for high-dimensional data
5. **Perceptron** - Single-layer neural network, linear classifier

**Evaluation Priority:** RECALL
- In cybersecurity, false negatives (attacks classified as benign) are more dangerous than false positives
- We prioritize recall to minimize undetected attacks

In [ ]:
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Perceptron': Perceptron(max_iter=1000, random_state=42)
}

# Train and evaluate each model
results = {}

print("="*80)
print("MODEL TRAINING AND EVALUATION")
print("="*80)

for name, model in models.items():
    print(f"\n{'='*80}")
    print(f"Training: {name}")
    print('='*80)

    # Time the training
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    training_time = time.time() - start_time

    # Make predictions
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)  # MOST IMPORTANT!
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'training_time': training_time
    }

    print(f"Training time: {training_time:.2f} seconds")
    print(f"Accuracy:      {accuracy:.4f}")
    print(f"Precision:     {precision:.4f}")
    print(f"Recall:        {recall:.4f} ← PRIORITY METRIC (detect attacks)")
    print(f"F1-Score:      {f1:.4f}")

print("\n" + "="*80)
print("ALL MODELS TRAINED SUCCESSFULLY")
print("="*80)

In [ ]:
# Create comparison table
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Time (s)':<10}")
print("-"*80)

for name, metrics in results.items():
    print(f"{name:<25} {metrics['accuracy']:<12.4f} {metrics['precision']:<12.4f} "
          f"{metrics['recall']:<12.4f} {metrics['f1_score']:<12.4f} {metrics['training_time']:<10.2f}")

# Find best model by recall (our priority metric)
best_model_name = max(results, key=lambda x: results[x]['recall'])
best_recall = results[best_model_name]['recall']

print("\n" + "="*80)
print(f"BEST MODEL (by Recall): {best_model_name}")
print(f"   Recall: {best_recall:.4f} ({best_recall*100:.2f}% of attacks detected)")
print("="*80)

In [ ]:
# Plot confusion matrices for top 2 models by recall
top_2_models = sorted(results.items(), key=lambda x: x[1]['recall'], reverse=True)[:2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, metrics) in enumerate(top_2_models):
    cm = metrics['confusion_matrix']

    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                cbar_kws={'label': 'Count'})

    axes[idx].set_title(f'{name}\nRecall: {metrics["recall"]:.4f} | Accuracy: {metrics["accuracy"]:.4f}',
                       fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    axes[idx].set_ylabel('True Label', fontsize=11)
    axes[idx].set_xticklabels(['BENIGN (0)', 'ATTACK (1)'])
    axes[idx].set_yticklabels(['BENIGN (0)', 'ATTACK (1)'])

    # Add text explanations
    tn, fp, fn, tp = cm.ravel()
    axes[idx].text(0.5, -0.15,
                  f'TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}',
                  ha='center', transform=axes[idx].transAxes, fontsize=9)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("CONFUSION MATRIX EXPLANATION:")
print("="*60)
print("TN (True Negative):  Correctly identified BENIGN traffic")
print("FP (False Positive): BENIGN traffic incorrectly flagged as ATTACK")
print("FN (False Negative): ATTACK traffic missed (DANGEROUS!)")
print("TP (True Positive):  Correctly identified ATTACK traffic")

In [ ]:
# Feature importance analysis (for Random Forest)
if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    feature_importance = pd.DataFrame({
        'feature': X_train_scaled.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("="*60)
    print("TOP 10 MOST IMPORTANT FEATURES (Random Forest)")
    print("="*60)
    print(feature_importance.head(10).to_string(index=False))

    # Plot top 15 features
    plt.figure(figsize=(10, 6))
    top_15 = feature_importance.head(15)
    plt.barh(range(len(top_15)), top_15['importance'])
    plt.yticks(range(len(top_15)), top_15['feature'])
    plt.xlabel('Importance Score', fontsize=11)
    plt.ylabel('Feature', fontsize=11)
    plt.title('Top 15 Most Important Features for DoS Attack Detection',
              fontsize=13, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Detailed analysis of best model
best_model = results[best_model_name]
cm = best_model['confusion_matrix']
tn, fp, fn, tp = cm.ravel()

print("="*80)
print(f"DETAILED ANALYSIS: {best_model_name}")
print("="*80)

print(f"\nPerformance Metrics:")
print(f"  Accuracy:  {best_model['accuracy']:.4f} ({best_model['accuracy']*100:.2f}%)")
print(f"  Precision: {best_model['precision']:.4f} ({best_model['precision']*100:.2f}%)")
print(f"  Recall:    {best_model['recall']:.4f} ({best_model['recall']*100:.2f}%)  ← PRIORITY")
print(f"  F1-Score:  {best_model['f1_score']:.4f}")

print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives  (TN): {tn:,} - Benign correctly identified")
print(f"  False Positives (FP): {fp:,} - Benign flagged as attack")
print(f"  False Negatives (FN): {fn:,} - Attacks MISSED")
print(f"  True Positives  (TP): {tp:,} - Attacks correctly detected")

print(f"\nSecurity Analysis:")
print(f"  Total attacks in test set: {tp + fn:,}")
print(f"  Attacks detected:          {tp:,} ({tp/(tp+fn)*100:.2f}%)")
print(f"  Attacks missed:            {fn:,} ({fn/(tp+fn)*100:.2f}%)")
print(f"  False alarm rate:          {fp/(tn+fp)*100:.2f}%")

print(f"\nRecommendation:")
if best_model['recall'] >= 0.95:
    print(f"  Excellent recall ({best_model['recall']*100:.1f}%) - Suitable for deployment")
elif best_model['recall'] >= 0.90:
    print(f"  Good recall ({best_model['recall']*100:.1f}%) - Consider hyperparameter tuning")
else:
    print(f"  Moderate recall ({best_model['recall']*100:.1f}%) - Needs improvement before deployment")

In [ ]:
print("="*80)
print("PROJECT CONCLUSIONS")
print("="*80)

print("\n1. DATA PROCESSING:")
print(f"   - Processed {len(df):,} network flows")
print(f"   - Removed Heartbleed (non-DoS attack)")
print(f"   - Binary classification: BENIGN vs ATTACK")
print(f"   - Final dataset: {X_train.shape[0] + X_test.shape[0]:,} samples")
print(f"   - Feature reduction: {X.shape[1]} → {X_train_scaled.shape[1]} features")

print("\n2. MODEL PERFORMANCE:")
for name, metrics in sorted(results.items(), key=lambda x: x[1]['recall'], reverse=True):
    print(f"   {name}:")
    print(f"      Recall: {metrics['recall']:.4f} | Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1_score']:.4f}")

print(f"\n3. BEST MODEL: {best_model_name}")
print(f"   - Successfully detects {best_model['recall']*100:.2f}% of DoS attacks")
print(f"   - Overall accuracy: {best_model['accuracy']*100:.2f}%")
print(f"   - Training time: {best_model['training_time']:.2f} seconds")

print("\n4. REAL-WORLD IMPLICATIONS:")
print(f"   - Can be deployed for real-time DoS attack detection")
print(f"   - Minimizes false negatives (missed attacks)")
print(f"   - Suitable for network security monitoring")

print("\n5. FUTURE IMPROVEMENTS:")
print("   - Hyperparameter tuning for better performance")
print("   - Multi-class classification (identify specific DoS types)")
print("   - Real-time streaming data pipeline")
print("   - Periodic model retraining with new attack patterns")

print("\n" + "="*80)
print("PROJECT COMPLETE ✓")
print("="*80)